In [15]:
%pip install pymysql

Note: you may need to restart the kernel to use updated packages.


In [16]:
import pandas as pd
import re
import json

In [17]:
df = pd.read_csv("/KC_PET_ACP_CTLSTT_LC_DATA_2023.csv")

In [18]:
def parse_pet_charge(text):
    """
    return:
      (charge_type, min_price, max_price)
    """

    if text is None:
        return ("UNKNOWN", None, None)

    t = str(text).strip()

    # ===== 무료 =====
    if any(k in t for k in ["없음", "무료"]):
        return ("FREE", 0, 0)

    # ===== 변동 / 불명 =====
    if any(k in t for k in ["변동", "문의", "상담"]):
        return ("UNKNOWN", None, None)

    # 숫자 추출
    nums = list(map(int, re.findall(r"\d+", t)))

    if not nums:
        return ("UNKNOWN", None, None)

    min_price = min(nums)
    max_price = max(nums)

    # ===== 조건부 =====
    if any(k in t for k in ["kg", "마리", "견", "소형", "중형", "대형", "이하", "이상"]):
        return ("CONDITIONAL", min_price, max_price)

    # ===== 범위 =====
    if "~" in t or "-" in t:
        return ("RANGE", min_price, max_price)

    # ===== 단일 금액 =====
    if len(nums) == 1:
        return ("FIXED", min_price, min_price)

    # fallback
    return ("UNKNOWN", min_price, max_price)


In [19]:
def map_charge_filter(min_price, max_price):
    if min_price == 0 and max_price == 0:
        return "FREE"

    if max_price is None:
        return None

    if max_price <= 5000:
        return "UNDER_5000"

    if max_price <= 10000:
        return "UNDER_10000"

    return "OVER_10000"

In [20]:
import pymysql

# DB 연결
conn = pymysql.connect(
    host="localhost",
    user="root",
    password="root",
    db="miniproject",
    charset="utf8mb4"
)
cursor = conn.cursor()

# place 테이블에서 id, fclty_nm, ADDR 가져오기
cursor.execute("SELECT id, fclty_nm, LNM_ADDR FROM place")
place_map = {
    (str(name).strip(), str(addr).strip()): pid
    for pid, name, addr in cursor.fetchall()
}

In [21]:
sql = """
INSERT INTO chargefilterdata (
    place_id,
    PET_CHARGE_TYPE,
    PET_CHARGE_MIN,
    PET_CHARGE_MAX,
    PET_ACP_ADIT_CHRGE_VALUE
) VALUES (%s, %s, %s, %s, %s)
"""

for _, r in df.iterrows():
    place_name = str(r["FCLTY_NM"]).strip()
    place_addr = str(r["LNM_ADDR"]).strip()
    charge_text = r["PET_ACP_ADIT_CHRGE_VALUE"]

    place_id = place_map.get((place_name, place_addr))
    if place_id is None:
        print(f"❗ place_id 매핑 실패: {place_name}")
        continue

    charge_type, min_price, max_price = parse_pet_charge(charge_text)

    cursor.execute(
        sql,
        (place_id, charge_type, min_price, max_price, charge_text)
    )

conn.commit()
conn.close()
